## Étape 11 : évaluation

Ce notebook fait passer de vrais tickets dans le graphe et mesure ce que demande l'énoncé : réponses correctes, pertinence des escalades, qualité des sources citées, erreurs sur les prix.

Le système est évalué dans **deux réglages** de la vérification des passages :
- **strict** : le LLM valide chaque passage sur deux critères, et seuls les passages validés servent à rédiger ;
- **souple** : le LLM rend un avis global, et tous les passages retrouvés servent à rédiger.

Trois parties :
- **Partie A, répétition** : 40 tickets de `KB_QA`, fiche cachée. Déjà faite, elle a servi à régler le système.
- **Partie B, questions récurrentes** : 40 questions de `KB_QA` reformulées, fiche présente.
- **Partie C, évaluation finale** : les 100 tickets de `EVAL_TICKETS`. À lancer **une seule fois**.

Chaque résultat est enregistré dès qu'il est obtenu. Une cellule interrompue reprend là où elle s'était arrêtée.

In [ ]:
!pip install langgraph langchain-openai rank_bm25

In [ ]:
import os
import sys

import pandas as pd
from snowflake.snowpark.context import get_active_session

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.append(ROOT)

import src.nodes.doc_search as doc_search_module
from src.evaluation import compare, load_done, paraphrase_tickets, rejudge, run_all, summarize
from src.graph import build_graph
from src.llm import make_llm
from src.nodes.classify import load_examples
from src.tools.retrieval import search_docs

session = get_active_session()
llm = make_llm()

MODES = ("strict", "souple")
GRAPHS = {mode: build_graph(llm=llm, session=session, verification=mode) for mode in MODES}

# Réglage retenu pour le système, décidé AVANT l'évaluation finale.
# L'autre réglage est mesuré à titre de comparaison.
RETAINED = "strict"
print("Réglage retenu :", RETAINED)

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 200)

### Cacher sa propre fiche à la recherche

Sert seulement à la répétition, où les tickets viennent de `KB_QA`. Dans les parties B et C, rien n'est caché.

In [ ]:
CURRENT = {"id": None}


def search_without_own_record(session, question, top_k=5, **kwargs):
    exclude = [CURRENT["id"]] if CURRENT["id"] else None
    return search_docs(session, question, top_k=top_k, exclude_ids=exclude, **kwargs)


doc_search_module.search_docs = search_without_own_record

kb = session.table("PROJECT_DB.PUBLIC.KB_QA").to_pandas()
example_ids = {e["id"] for e in load_examples(session)}
pool = kb[~kb["ID"].isin(example_ids)]

## Partie A : répétition (déjà faite)

40 tickets de `KB_QA` dont la fiche était cachée, passés une fois dans chaque réglage le 6 octobre. Cette cellule ne repasse aucun ticket : elle relit les deux tables de résultats et les compare. Les réponses du réglage souple sont rejugées une fois avec la grille actuelle, puis gardées.

In [ ]:
def rehearsal():
    strict = load_done(session, "PROJECT_DB.PUBLIC.EVAL_RESULTS_DEV_V2")
    souple = load_done(session, "PROJECT_DB.PUBLIC.EVAL_RESULTS_DEV_REJUDGED")
    if not len(souple):
        first = load_done(session, "PROJECT_DB.PUBLIC.EVAL_RESULTS_DEV")
        if not len(first):
            return None
        souple = rejudge(llm, first)
        session.write_pandas(souple, "EVAL_RESULTS_DEV_REJUDGED", database="PROJECT_DB", schema="PUBLIC",
                             auto_create_table=True, overwrite=True)
    return {"souple": souple, "strict": strict} if len(strict) else None


dev_runs = rehearsal()
print(compare(dev_runs).to_string(index=False) if dev_runs else "Tables de répétition introuvables")

## Partie B : questions récurrentes

Une question récurrente est une question dont la base connaît déjà la réponse, posée à nouveau avec d'autres mots. C'est le cas visé par l'objectif de 50 à 60 % de l'énoncé.

On prend 40 fiches « Resolved » de `KB_QA`, le LLM reformule chaque question, et le ticket reformulé passe dans le graphe, une fois par réglage. La fiche d'origine reste dans la base : le système doit la retrouver et donner la même solution.

Limite : une reformulation écrite par un LLM reste proche de l'original. Le résultat est donc optimiste.

In [ ]:
recurring_source = pool[pool["STATUS"] == "Resolved"].sample(40, random_state=11).sort_values("ID").reset_index(drop=True)
recurring = paraphrase_tickets(session, llm, recurring_source, "PROJECT_DB.PUBLIC.EVAL_RECURRING_TICKETS")

print(recurring[["ID", "ORIGINAL_QUESTION", "QUESTION"]].head(3).to_string(index=False))

In [ ]:
rec_runs = {}
for mode in MODES:
    print(f"\n===== Réglage {mode} =====")
    rec_runs[mode] = run_all(session, GRAPHS[mode], llm, recurring, f"PROJECT_DB.PUBLIC.EVAL_RESULTS_RECURRING_{mode.upper()}",
                             before_ticket=lambda ticket_id: CURRENT.update(id=None))

In [ ]:
print(compare(rec_runs).to_string(index=False))

In [ ]:
for mode in MODES:
    res = rec_runs[mode]
    answered = res[res["DECISION"] == "repondre"]
    own = answered.apply(lambda r: r["ID"] in r["SOURCES_FOUND"], axis=1)
    print(f"\n===== Réglage {mode} =====")
    print(summarize(res)["by_rule"].to_string())
    print(f"Fiche d'origine parmi les passages retrouvés : {int(own.sum())} / {len(answered)} réponses")
    print(answered[answered["VERDICT"] != "correct"][["ID", "VERDICT", "VERDICT_REASON"]].to_string(index=False))

In [ ]:
%%sql -r dataframe_1
-- TRUNCATE TABLE PROJECT_DB.PUBLIC.EVAL_RESULTS_STRICT;
-- TRUNCATE TABLE PROJECT_DB.PUBLIC.EVAL_RESULTS_MODERATE;
-- TRUNCATE TABLE PROJECT_DB.PUBLIC.EVAL_RESULTS_RELAXED;

## Partie C : évaluation finale sur `EVAL_TICKETS`

Les 100 tickets réservés, dont les fiches ne sont pas dans la base : ce sont des problèmes nouveaux pour le système. Chaque ticket passe une fois par réglage.

À lancer une seule fois. Passe `RUN_FINAL` à `True` seulement quand le système est figé. Compte environ quarante minutes pour les deux réglages.

In [ ]:
RUN_FINAL = True

final_runs = None
if RUN_FINAL:
    eval_tickets = session.table("PROJECT_DB.PUBLIC.EVAL_TICKETS").to_pandas().sort_values("ID").reset_index(drop=True)
    final_runs = {}
    for mode in MODES:
        print(f"\n===== Réglage {mode} =====")
        final_runs[mode] = run_all(session, GRAPHS[mode], llm, eval_tickets, f"PROJECT_DB.PUBLIC.EVAL_RESULTS_{mode.upper()}",
                                   before_ticket=lambda ticket_id: CURRENT.update(id=None))
else:
    print("Évaluation finale non lancée : RUN_FINAL vaut False.")

In [ ]:
print(compare(final_runs).to_string(index=False) if final_runs else "Évaluation finale non lancée")

In [ ]:
if final_runs:
    for mode in MODES:
        res = final_runs[mode]
        answered = res[res["DECISION"] == "repondre"]
        print(f"\n===== Réglage {mode} =====")
        print(summarize(res)["by_rule"].to_string())
        print()
        print(summarize(res)["by_theme"].to_string())
        print()
        print(answered[["ID", "STATUS", "PRICE_CASE", "VERDICT", "VERDICT_REASON"]].to_string(index=False))
else:
    print("Évaluation finale non lancée")

In [ ]:
if final_runs:
    res = final_runs[RETAINED]
    to_review = res[
        res["VERDICT"].isin(["incorrect", "partiel"])
        | ((res["DECISION"] == "repondre") & (res["PRICE_CASE"] | (res["STATUS"] == "Out of Scope")))
    ][["ID", "STATUS", "PRICE_CASE", "VERDICT", "VERDICT_REASON", "QUESTION", "REFERENCE", "ANSWER"]]
else:
    to_review = "Évaluation finale non lancée"
to_review